In [1]:
print('hello')

hello


In [2]:
%pip install rank_bm25

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [26]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
# import faiss
# from llama_index.vector_stores.faiss import FaissVectorStore
from langchain.vectorstores import FAISS
from langchain.retrievers import ContextualCompressionRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.retrievers import BM25Retriever, EnsembleRetriever

In [27]:
import os
from getpass import getpass

HF_token = getpass()
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HF_token

In [28]:
#loading the pdfs
from langchain_community.document_loaders.pdf import PyPDFDirectoryLoader
from langchain.schema.document import Document
def load_documents():
    document_loader = PyPDFDirectoryLoader('pdfs')
    return document_loader.load()
# documents = load_documents()
# print(documents[0])

#split the documents
def split_documents(documents: list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False,
    )
    return text_splitter.split_documents(documents)
documents = load_documents()
chunks = split_documents(documents)
print(chunks[0])

page_content='www.pes.edu PESU C-ISFCR & PESU C-IoT, PES UniversityPESU RESEARCH\nREVIEW\nVolume 1\n2018 - September 2021' metadata={'source': 'pdfs\\Research_Vol_2021.pdf', 'page': 0}


In [29]:
embeddings=HuggingFaceInferenceAPIEmbeddings(
    api_key=HF_token,
    model_name='BAAI/bge-base-en-v1.5'
)

In [30]:
from llama_index.core import VectorStoreIndex

In [38]:
vectorstore = FAISS.from_documents(chunks, embeddings)
index = VectorStoreIndex.from_documents(vectorstore)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api-inference.huggingface.co:443
Starting new HTTPS connection (1): api-inference.huggingface.co:443
Starting new HTTPS connection (1): api-inference.huggingface.co:443
DEBUG:urllib3.connectionpool:https://api-inference.huggingface.co:443 "POST /pipeline/feature-extraction/BAAI/bge-base-en-v1.5 HTTP/1.1" 200 7996796
https://api-inference.huggingface.co:443 "POST /pipeline/feature-extraction/BAAI/bge-base-en-v1.5 HTTP/1.1" 200 7996796
https://api-inference.huggingface.co:443 "POST /pipeline/feature-extraction/BAAI/bge-base-en-v1.5 HTTP/1.1" 200 7996796


TypeError: 'FAISS' object is not iterable

In [32]:
retriever_vectordb = vectorstore.as_retriever(search_kwargs={"k": 5})
keyword_retriever = BM25Retriever.from_documents(chunks)#text_splits = chunks
keyword_retriever.k =  5
ensemble_retriever = EnsembleRetriever(retrievers=[retriever_vectordb,keyword_retriever],
                                       weights=[0.5, 0.5])

In [33]:
from langchain.llms import HuggingFaceHub
model=HuggingFaceHub(repo_id='HuggingFaceH4/zephyr-7b-alpha',
                     model_kwargs={"temperature":0.5,"max_new_tokens":512,"max_length":64}
)

DEBUG:urllib3.connectionpool:Resetting dropped connection: huggingface.co
Resetting dropped connection: huggingface.co
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "GET /api/models/HuggingFaceH4/zephyr-7b-alpha HTTP/1.1" 200 6939
https://huggingface.co:443 "GET /api/models/HuggingFaceH4/zephyr-7b-alpha HTTP/1.1" 200 6939


In [34]:
import os
# from llama_index.core import VectorStoreIndex

from llama_index.core import Settings
from llama_index.postprocessor.colbert_rerank import ColbertRerank

In [35]:
from langchain_community.embeddings.ollama import OllamaEmbeddings

from langchain_community.llms.ollama import Ollama
# Setup the OpenAI LLM
Settings.llm = Ollama(model="mistral:instruct")
Settings.embed_model = OllamaEmbeddings(model="nomic-embed-text")

In [36]:
# Optional: Set up debug logging to see exactly what llamaindex is doing
import logging
import sys
logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [37]:
colbert_reranker = ColbertRerank(
    top_n=5,
    model="colbert-ir/colbertv2.0",
    tokenizer="colbert-ir/colbertv2.0",
    keep_retrieval_score=True,
)

query_engine = vectorstore.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[colbert_reranker],
)

DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /colbert-ir/colbertv2.0/resolve/main/tokenizer_config.json HTTP/1.1" 200 0
https://huggingface.co:443 "HEAD /colbert-ir/colbertv2.0/resolve/main/tokenizer_config.json HTTP/1.1" 200 0
https://huggingface.co:443 "HEAD /colbert-ir/colbertv2.0/resolve/main/tokenizer_config.json HTTP/1.1" 200 0
DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /colbert-ir/colbertv2.0/resolve/main/config.json HTTP/1.1" 200 0
https://huggingface.co:443 "HEAD /colbert-ir/colbertv2.0/resolve/main/config.json HTTP/1.1" 200 0
https://huggingface.co:443 "HEAD /colbert-ir/colbertv2.0/resolve/main/config.json HTTP/1.1" 200 0


AttributeError: 'FAISS' object has no attribute 'as_query_engine'